In [ ]:
!git clone https://github.com/ZaryabRahman/Deep-Spec

Cloning into 'Deep-Spec'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 15 (delta 1), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 4.82 KiB | 1.20 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
!cd Deep-Spec

In [ ]:
import os
import sys

print("searching for your Deep-Spec source code...")

package_path = None
for root, dirs, files in os.walk('/content/Deep-Spec'):
    if 'core.py' in files and 'aggregation.py' in files:
        package_path = root
        break

if package_path is None:
    print("could not find 'core.py'")
else:
    print(f"found the code inside: {package_path}")

    parent_dir = os.path.dirname(package_path)
    actual_folder_name = os.path.basename(package_path)

    if actual_folder_name != 'deep_spec':
        new_path = os.path.join(parent_dir, 'deep_spec')
        print(f"renaming folder from '{actual_folder_name}' to 'deep_spec' so Python can read it...")
        os.rename(package_path, new_path)
        package_path = new_path

    init_file = os.path.join(package_path, '__init__.py')
    if not os.path.exists(init_file):
        print("missing __init__.py found. Creating it now...")
        with open(init_file, 'w') as f:
            f.write("from .core import DeepSpecExplainer\nfrom .aggregation import aggregate_layers\n")

    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

    print("\ninjecting into Python and importing...")

    try:
        from deep_spec import DeepSpecExplainer, aggregate_layers
        print(" `deep_spec` successfully imported and is ready to use!")
    except Exception as e:
        print(f"still failing with error: {e}")

In [ ]:
pip install -e /content/Deep-Spec

Obtaining file:///content/Deep-Spec
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for deep-spec (pyproject.toml) ... done
  Created wheel for deep-spec: filename=deep_spec-0.1.0-0.editable-py3-none-any.whl size=3067 sha256=94263baf67121a79df29cc6fa2db45ea6802e621075222bc1da152d96d00f14d
  Stored in directory: /tmp/pip-ephem-wheel-cache-fs8620k4/wheels/4b/d0/ef/7aeac8559e64b5316b6f965a6a02822d775996c9cb4364c805
Successfully built deep-spec


In [ ]:
%cd /content/Deep-Spec


/content/Deep-Spec


In [ ]:
!pip install .


Processing /content/Deep-Spec
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for deep-spec: filename=deep_spec-0.1.0-py3-none-any.whl size=4766 sha256=4f646fde3702feb845dd703367622ddf209bd10131daddfc0f09e0c3a80e3074
  Stored in directory: /root/.cache/pip/wheels/4b/d0/ef/7aeac8559e64b5316b6f965a6a02822d775996c9cb4364c805
Successfully built deep-spec


In [ ]:
"""
Deep-Spec Large-Scale Pointing Game Benchmark — BLIP
==================================================================================
The True Run: 1-to-1 integration using the published `deep_spec` package!
"""

import torch
import numpy as np
from scipy.linalg import eigh
from transformers import BlipProcessor, BlipForImageTextRetrieval
from datasets import load_dataset
from tqdm import tqdm
import warnings
import logging
import requests
import json
import os
from io import BytesIO
from PIL import Image

from deep_spec import DeepSpecExplainer, aggregate_layers

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP Foundation Model...")
MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()

GRID_SIZE = 24


ANNOTATION_FILE = "instances_val2017.json"
if not os.path.exists(ANNOTATION_FILE):
    print("Downloading MS-COCO validation bounding boxes...")
    import zipfile
    r = requests.get("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", stream=True)
    with open("annotations.zip", 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    with zipfile.ZipFile("annotations.zip", 'r') as zip_ref:
        zip_ref.extract("annotations/instances_val2017.json", ".")
    os.rename("annotations/instances_val2017.json", ANNOTATION_FILE)
    print("Download complete.")

with open(ANNOTATION_FILE, 'r') as f:
    coco_data = json.load(f)

categories = {cat['id']: cat['name'] for cat in coco_data['categories']}
image_to_objects = {}
for ann in coco_data['annotations']:
    img_id = ann['image_id']
    if img_id not in image_to_objects:
        image_to_objects[img_id] = []
    image_to_objects[img_id].append({
        'category_name': categories[ann['category_id']],
        'bbox': ann['bbox']
    })

dataset = load_dataset("phiyodr/coco2017", split="validation", streaming=True)

methods = ["Raw Attention", "Attention Rollout", "ViT-Grad-CAM", "Attn x Grad (Chefer)", "Deep-Spec (Ours)"]
hits = {m: 0 for m in methods}
total_evaluated = 0
NUM_EVALS = 1000

def extract_all_heatmaps(inputs, valid_indices, tgt_rel):
    heatmaps = {}

    attn_maps = []
    hooks = []
    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        _ = model(**inputs)

    for h in hooks:
        h.remove()

    all_layers_attn = torch.stack(attn_maps)

    # raw attention
    avg_attn = all_layers_attn.mean(dim=(0, 1, 2))
    avg_attn = avg_attn[valid_indices, 1:]

    hm_raw = avg_attn[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_raw = hm_raw / (hm_raw.max() + 1e-10)
    heatmaps["Raw Attention"] = hm_raw.numpy()


    T, I = all_layers_attn.shape[-2], all_layers_attn.shape[-1]
    rollout = torch.eye(T + I)

    for layer_attn in all_layers_attn:
        avg_heads = layer_attn.mean(dim=(0, 1))
        full = torch.zeros((T + I, T + I))
        full[:T, T:] = avg_heads
        full[T:, :T] = avg_heads.T
        full = 0.5 * full + 0.5 * torch.eye(T + I)
        rollout = rollout @ full

    text_to_img_rollout = rollout[:T, T:]
    filtered_rollout = text_to_img_rollout[valid_indices, 1:]

    hm_roll = filtered_rollout[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_roll = hm_roll / (hm_roll.max() + 1e-10)
    heatmaps["Attention Rollout"] = hm_roll.numpy()

    # DEEP-SPEC (Powered by your PyPI package!)
    # 1. first pepare tensor shape (remove Batch dimension so it becomes Layers x Heads x T x I)
    attn_4d = all_layers_attn.squeeze(1)

    # 2. package Function #1: Framework-agnostic aggregation
    A_global = aggregate_layers(attn_4d, method="mean")

    # 3. filter valid token text indices and slice out image CLS token (1:)
    A_filtered = A_global[valid_indices, 1:]

    # 4. package function #2: Low-Rank Spectral Partitioning Engine
    hm_ds = DeepSpecExplainer.compute_heatmap(
        attention_matrix=A_filtered,
        grid_size=(GRID_SIZE, GRID_SIZE),
        target_indices=tgt_rel,
        k=5,
        epsilon=1e-3
    )
    heatmaps["Deep-Spec (Ours)"] = hm_ds


    #Grad-CAM & Chefer)
    activations, gradients = None, None
    attn_maps_grad, grad_maps = [], []

    def forward_hook_gcam(module, inp, out):
        nonlocal activations
        activations = out[0].detach()

    def backward_hook_gcam(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0].detach()

    def forward_hook_chefer(module, inp, out):
        attn = inp[0]
        attn_maps_grad.append(attn)
        attn.retain_grad()
        attn.register_hook(lambda g: grad_maps.append(g))

    hooks_grad = []
    target_layer = model.vision_model.encoder.layers[-1]
    hooks_grad.append(target_layer.register_forward_hook(forward_hook_gcam))
    hooks_grad.append(target_layer.register_full_backward_hook(backward_hook_gcam))

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks_grad.append(module.register_forward_hook(forward_hook_chefer))

    model.zero_grad()
    outputs = model(**inputs)
    itm_score = outputs.itm_score[0, 1]
    itm_score.backward()

    for h in hooks_grad:
        h.remove()

    # ViT-Grad-CAM
    if gradients is not None and activations is not None:
        weights = gradients.mean(dim=(0, 1))
        cam = (weights * activations).sum(dim=-1).squeeze(0)
        cam = torch.relu(cam)[1:]
        cam = cam.reshape(GRID_SIZE, GRID_SIZE)
        cam = cam / (cam.max() + 1e-10)
        heatmaps["ViT-Grad-CAM"] = cam.detach().cpu().numpy()
    else:
        heatmaps["ViT-Grad-CAM"] = np.zeros((GRID_SIZE, GRID_SIZE))

    # Chefer
    if len(attn_maps_grad) > 0 and len(grad_maps) > 0:
        heatmap_chefer = torch.zeros_like(attn_maps_grad[0].mean(dim=(0, 1)))
        for attn, grad in zip(attn_maps_grad, grad_maps):
            heatmap_chefer += (attn * grad).mean(dim=(0, 1))

        heatmap_chefer = heatmap_chefer / len(attn_maps_grad)
        heatmap_chefer = heatmap_chefer[valid_indices, 1:]
        heatmap_chefer = torch.clamp(heatmap_chefer, min=0)

        hm_chefer = heatmap_chefer[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
        hm_chefer = hm_chefer / (hm_chefer.max() + 1e-10)
        heatmaps["Attn x Grad (Chefer)"] = hm_chefer.detach().cpu().numpy()
    else:
        heatmaps["Attn x Grad (Chefer)"] = np.zeros((GRID_SIZE, GRID_SIZE))

    return heatmaps

print(f"Starting Golden Run: Large-Scale BLIP Pointing Game over {NUM_EVALS} objects...\n")
iterator = iter(dataset)

with tqdm(total=NUM_EVALS, desc="Evaluating", unit="obj") as pbar:
    while total_evaluated < NUM_EVALS:
        try:
            data = next(iterator)
            image_id = data.get('image_id')

            if image_id not in image_to_objects: continue
            objects = image_to_objects[image_id]

            if 'image' in data and data['image'] is not None:
                image = data['image'].convert("RGB")
            elif 'coco_url' in data:
                response = requests.get(data['coco_url'], timeout=10)
                image = Image.open(BytesIO(response.content)).convert("RGB")
            else: continue

            orig_W, orig_H = image.size

            caption = data['captions'][0] if isinstance(data.get('captions'), list) else data.get('text', "")
            if not caption: continue

            target_noun = None
            for obj in objects:
                if obj['category_name'].lower() in caption.lower():
                    target_noun = obj['category_name']
                    break

            if not target_noun: continue
            all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]

            inputs = processor(images=image, text=caption, return_tensors="pt").to(device)

            tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
            special_ids = processor.tokenizer.all_special_ids
            valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]

            target_indices = []
            for i, t in enumerate(tokens):
                if i in valid_indices:
                    clean_token = t.lower().replace("##", "")
                    if any(word in clean_token or clean_token in word for word in target_noun.lower().split()):
                        target_indices.append(i)

            if not target_indices: continue

            tgt_rel = [valid_indices.index(idx) for idx in target_indices]

            heatmaps = extract_all_heatmaps(inputs, valid_indices, tgt_rel)
            if heatmaps is None: continue

            for m_name in methods:
                hm = heatmaps[m_name]
                if hm.max() == 0: continue

                max_idx = np.argmax(hm)
                max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)

                rel_x = (max_x_patch + 0.5) / GRID_SIZE
                rel_y = (max_y_patch + 0.5) / GRID_SIZE
                pixel_x = rel_x * orig_W
                pixel_y = rel_y * orig_H

                hit = False
                for bbox in all_target_bboxes:
                    x_min, y_min, bbox_w, bbox_h = bbox
                    x_max, y_max = x_min + bbox_w, y_min + bbox_h
                    if (x_min <= pixel_x <= x_max) and (y_min <= pixel_y <= y_max):
                        hit = True
                        break

                if hit:
                    hits[m_name] += 1

            total_evaluated += 1
            pbar.update(1)

            if total_evaluated % 100 == 0:
                pbar.write(f"\n--- Pointing Game Accuracy ({total_evaluated}/{NUM_EVALS}) ---")
                for m_name in methods:
                    acc = (hits[m_name] / total_evaluated) * 100
                    pbar.write(f"{m_name:<25} | {acc:.1f}%")
                pbar.write("-" * 50)

        except StopIteration:
            break
        except Exception as e:
            continue

print("\n" + "=" * 60)
print(" FINAL POINTING GAME ACCURACY (Zero-Shot Localization) ".center(60))
print("=" * 60)
print(f"{'Method':<25} | {'Hit Accuracy (%)':<20}")
print("-" * 60)
for m_name in methods:
    acc = (hits[m_name] / total_evaluated) * 100
    prefix = ">> " if "Deep-Spec" in m_name else "   "
    print(f"{prefix}{m_name:<22} | {acc:.2f}%")
print("=" * 60)

Loading BLIP Foundation Model...


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...


--- Pointing Game Accuracy (800/1000) ---
Raw Attention             | 12.8%
Attention Rollout         | 12.8%
ViT-Grad-CAM              | 12.9%
Attn x Grad (Chefer)      | 66.4%
Deep-Spec (Ours)          | 68.1%
--------------------------------------------------


Evaluating:  90%|█████████ | 900/1000 [14:49<01:15,  1.32obj/s]


--- Pointing Game Accuracy (900/1000) ---
Raw Attention             | 12.6%
Attention Rollout         | 12.4%
ViT-Grad-CAM              | 12.8%
Attn x Grad (Chefer)      | 66.7%
Deep-Spec (Ours)          | 69.0%
--------------------------------------------------


Evaluating: 100%|██████████| 1000/1000 [16:24<00:00,  1.02obj/s]


--- Pointing Game Accuracy (1000/1000) ---
Raw Attention             | 12.2%
Attention Rollout         | 12.1%
ViT-Grad-CAM              | 12.7%
Attn x Grad (Chefer)      | 66.5%
Deep-Spec (Ours)          | 68.5%
--------------------------------------------------

   FINAL POINTING GAME ACCURACY (Zero-Shot Localization)    
Method                    | Hit Accuracy (%)    
------------------------------------------------------------
   Raw Attention          | 12.20%
   Attention Rollout      | 12.10%
   ViT-Grad-CAM           | 12.70%
   Attn x Grad (Chefer)   | 66.50%
>> Deep-Spec (Ours)       | 68.50%
